In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec

# --- Load & Preprocess ---
dataset_path = '../Email Dataset/Clean_Emails_Dataset.csv'
df = pd.read_csv(dataset_path)
df = df.dropna(subset=['Email Text', 'Email Type'])
df['label_num'] = df['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

X = df['Email Text'].astype(str)
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- Folder to save models ---
os.makedirs('../Models', exist_ok=True)

# --- TF-IDF Vectorizer (shared by LR and NB) ---
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

with open('../Models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print('Saved: tfidf_vectorizer.pkl')

# --- Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

with open('../Models/lr_model.pkl', 'wb') as f:
    pickle.dump(lr_model, f)
print('Saved: lr_model.pkl')

# --- Naive Bayes ---
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

with open('../Models/nb_model.pkl', 'wb') as f:
    pickle.dump(nb_model, f)
print('Saved: nb_model.pkl')

# --- Word2Vec + Random Forest ---
X_train_tokens = [text.split() for text in X_train]

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
w2v_model.save('../Models/w2v_model.model')
print('Saved: w2v_model.model')

def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

X_train_w2v = np.array([get_avg_vector(tokens, w2v_model) for tokens in X_train_tokens])

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_w2v, y_train)

with open('../Models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print('Saved: rf_model.pkl')

print('\nTraditional models trained and saved to ../Models/')

In [ ]:
import transformers
from packaging import version

print('transformers:', transformers.__version__)

if version.parse(transformers.__version__) >= version.parse('5.0.0'):
    raise RuntimeError(
        "This notebook expects TensorFlow-compatible Hugging Face APIs from transformers 4.x. "
        "Please run: %pip install 'transformers==4.46.3' tf-keras"
    )

print('Version check passed: transformers 4.x detected.')

In [ ]:
import os
import numpy as np
import tensorflow as tf
from transformers import DistilBertTokenizerFast, TFAutoModelForSequenceClassification


tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='np'
)

train_labels = y_train.reset_index(drop=True).to_numpy(dtype=np.int32)

train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': train_encodings['input_ids'],
        'attention_mask': train_encodings['attention_mask']
    },
    train_labels
)).shuffle(1000).batch(8)

distilbert_model = TFAutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy('accuracy')]

distilbert_model.compile(optimizer=optimizer, loss=loss, metrics=metrics)
distilbert_model.fit(train_dataset, epochs=2)

distilbert_save_dir = '../Models/distilbert_model'
os.makedirs(distilbert_save_dir, exist_ok=True)
distilbert_model.save_pretrained(distilbert_save_dir)
tokenizer.save_pretrained(distilbert_save_dir)
print('Saved: distilbert_model/')

In [ ]:
import numpy as np
import pickle
import tensorflow as tf
from gensim.models import Word2Vec
from transformers import DistilBertTokenizerFast, TFAutoModelForSequenceClassification

# --- Load all models ---
with open('../Models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)

with open('../Models/lr_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)

with open('../Models/nb_model.pkl', 'rb') as f:
    nb_model = pickle.load(f)

with open('../Models/rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

w2v_model = Word2Vec.load('../Models/w2v_model.model')

distilbert_dir = '../Models/distilbert_model'
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained(distilbert_dir)
distilbert_model = TFAutoModelForSequenceClassification.from_pretrained(distilbert_dir)

# --- Helper ---
def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

def label(pred):
    return '🚨 Phishing' if pred == 1 else '✅ Safe'

# --- Main prediction function ---
def predict_email(email_text):
    print('\n' + '=' * 50)
    print('📧 Analyzing Email...')
    print('=' * 50)

    # TF-IDF transform
    tfidf_vec = tfidf_vectorizer.transform([email_text])

    # Word2Vec transform
    tokens = email_text.split()
    w2v_vec = np.array([get_avg_vector(tokens, w2v_model)])

    # DistilBERT transform
    bert_inputs = distilbert_tokenizer(
        [email_text],
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors='tf'
    )

    # --- Logistic Regression ---
    lr_pred = lr_model.predict(tfidf_vec)[0]
    lr_conf = lr_model.predict_proba(tfidf_vec)[0][lr_pred] * 100
    print(f'\n[Logistic Regression]  {label(lr_pred)}  ({lr_conf:.1f}% confident)')

    # --- Naive Bayes ---
    nb_pred = nb_model.predict(tfidf_vec)[0]
    nb_conf = nb_model.predict_proba(tfidf_vec)[0][nb_pred] * 100
    print(f'[Naive Bayes]          {label(nb_pred)}  ({nb_conf:.1f}% confident)')

    # --- Random Forest ---
    rf_pred = rf_model.predict(w2v_vec)[0]
    rf_conf = rf_model.predict_proba(w2v_vec)[0][rf_pred] * 100
    print(f'[Random Forest]        {label(rf_pred)}  ({rf_conf:.1f}% confident)')

    # --- DistilBERT ---
    outputs = distilbert_model(bert_inputs)
    probs = tf.nn.softmax(outputs.logits, axis=1).numpy()[0]
    db_pred = int(np.argmax(probs))
    db_conf = float(probs[db_pred] * 100)
    print(f'[DistilBERT]           {label(db_pred)}  ({db_conf:.1f}% confident)')

    # --- Majority Vote ---
    votes = [lr_pred, nb_pred, rf_pred, db_pred]
    final = 1 if sum(votes) >= 2 else 0
    print(f'\n>>> Majority Verdict:  {label(final)}')
    print('=' * 50)


# --- Try it out ---
email = """
Dear user, your account has been suspended.
Click here immediately to verify your identity and avoid permanent ban: http://secure-login-verify.com
"""

predict_email(email)

In [ ]:
import re
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    mean_squared_error,
    confusion_matrix,
)


import pickle
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from transformers import DistilBertTokenizerFast, TFAutoModelForSequenceClassification

# Rebuild test split if case X_test/y_test are not in memory yet.
dataset_path = '../Email Dataset/Clean_Emails_Dataset.csv'
df_eval = pd.read_csv(dataset_path)
df_eval = df_eval.dropna(subset=['Email Text', 'Email Type'])
df_eval['label_num'] = df_eval['Email Type'].map({'Safe Email': 0, 'Phishing Email': 1})

X_eval = df_eval['Email Text'].astype(str)
y_eval = df_eval['label_num']

_, X_test, _, y_test = train_test_split(
    X_eval, y_eval, test_size=0.2, random_state=42, stratify=y_eval
)

# Load saved artifacts.
with open('../Models/tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
with open('../Models/lr_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)
with open('../Models/nb_model.pkl', 'rb') as f:
    nb_model = pickle.load(f)
with open('../Models/rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

w2v_model = None
try:
    w2v_model = Word2Vec.load('../Models/w2v_model.model')
except Exception as e:
    
    try:
        with open('../Models/w2v_model.model', 'rb') as f:
            w2v_model = pickle.load(f)
        print(f'Warning: Loaded Word2Vec via pickle fallback ({type(w2v_model).__name__}).')
    except Exception as e2:
        print('Warning: Could not load Word2Vec model. RandomForest/Ensemble will be skipped.')
        print(f'  Word2Vec.load error: {e}')
        print(f'  pickle.load error: {e2}')

import gc

distilbert_dir = '../Models/distilbert_model'
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained(distilbert_dir)


tf.keras.backend.clear_session()
gc.collect()

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

distilbert_model = TFAutoModelForSequenceClassification.from_pretrained(distilbert_dir)


def get_avg_vector(tokens, model):
    if model is None:
        return np.zeros(100)

    
    kv = model.wv if hasattr(model, 'wv') else model
    vectors = [kv[word] for word in tokens if word in kv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)


def _safe_auc(y_true, y_score):
    """Return ROC-AUC when available; otherwise NaN (e.g., single-class edge cases)."""
    try:
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan


def eval_binary_model(name, y_true, y_pred, y_prob):
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'ROC_AUC': _safe_auc(y_true, y_prob),
    }
    cm = confusion_matrix(y_true, y_pred)
    return metrics, cm



X_test_tfidf = tfidf_vectorizer.transform(X_test)

lr_pred = lr_model.predict(X_test_tfidf)
lr_prob = lr_model.predict_proba(X_test_tfidf)[:, 1]

nb_pred = nb_model.predict(X_test_tfidf)
nb_prob = nb_model.predict_proba(X_test_tfidf)[:, 1]


X_test_tokens = [text.split() for text in X_test.astype(str)]
rf_pred, rf_prob = None, None
if w2v_model is not None:
    X_test_w2v = np.array([get_avg_vector(tokens, w2v_model) for tokens in X_test_tokens])
    rf_pred = rf_model.predict(X_test_w2v)
    rf_prob = rf_model.predict_proba(X_test_w2v)[:, 1]


test_encodings = distilbert_tokenizer(
    X_test.astype(str).tolist(),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors='np'
)

def _predict_with_batch_size(model, encodings, batch_size, device_name):
    probs_out = []
    n = encodings['input_ids'].shape[0]
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_inputs = {
            'input_ids': tf.convert_to_tensor(encodings['input_ids'][start:end]),
            'attention_mask': tf.convert_to_tensor(encodings['attention_mask'][start:end]),
        }
        with tf.device(device_name):
            batch_logits = model(batch_inputs, training=False).logits
            batch_probs = tf.nn.softmax(batch_logits, axis=1).numpy()[:, 1]
        probs_out.append(batch_probs)
    return np.concatenate(probs_out, axis=0)


def distilbert_predict_proba_adaptive(model, encodings):
    
    has_gpu = len(tf.config.list_physical_devices('GPU')) > 0
    if has_gpu:
        for bs in [16, 8, 4, 2, 1]:
            try:
                print(f'DistilBERT inference on GPU with batch_size={bs}')
                return _predict_with_batch_size(model, encodings, batch_size=bs, device_name='/GPU:0')
            except tf.errors.ResourceExhaustedError:
                print(f'GPU OOM at batch_size={bs}, trying smaller batch...')
                tf.keras.backend.clear_session()
                gc.collect()

    print('Falling back to CPU inference (stable, slower).')
    return _predict_with_batch_size(model, encodings, batch_size=8, device_name='/CPU:0')


db_prob = distilbert_predict_proba_adaptive(distilbert_model, test_encodings)
db_pred = (db_prob >= 0.5).astype(int)

# Majority vote ensemble (with available models only)
pred_list = [lr_pred, nb_pred, db_pred]
prob_list = [lr_prob, nb_prob, db_prob]
if rf_pred is not None:
    pred_list.append(rf_pred)
    prob_list.append(rf_prob)

stacked_votes = np.vstack(pred_list).T
required_votes = int(np.ceil(len(pred_list) / 2.0))
ensemble_pred = (stacked_votes.sum(axis=1) >= required_votes).astype(int)
ensemble_prob = np.mean(np.vstack(prob_list), axis=0)


results = []
conf_mats = {}

eval_rows = [
    ('Logistic Regression', lr_pred, lr_prob),
    ('Naive Bayes', nb_pred, nb_prob),
    ('DistilBERT', db_pred, db_prob),
]
if rf_pred is not None:
    eval_rows.append(('Random Forest (Word2Vec)', rf_pred, rf_prob))

eval_rows.append(('Majority Ensemble', ensemble_pred, ensemble_prob))

for model_name, pred, prob in eval_rows:
    m, cm = eval_binary_model(model_name, y_test, pred, prob)
    results.append(m)
    conf_mats[model_name] = cm

metrics_df = pd.DataFrame(results).sort_values(by='F1', ascending=False)
print('=== Model Performance on Test Set ===')
display(metrics_df)

print('\n=== Confusion Matrices ===')
for name, cm in conf_mats.items():
    print(f'\n{name}:')
    print(cm)



bias_df = pd.DataFrame({
    'text': X_test.astype(str).values,
    'y_true': y_test.values,
    'y_pred': ensemble_pred,
    'y_prob': ensemble_prob,
})

bias_df['len_chars'] = bias_df['text'].str.len()
bias_df['has_url'] = bias_df['text'].str.contains(r'https?://|www\.', case=False, regex=True)
bias_df['has_urgent_terms'] = bias_df['text'].str.contains(
    r'urgent|immediately|verify|suspended|password|bank|click',
    case=False,
    regex=True,
)


bias_df['length_bucket'] = pd.cut(
    bias_df['len_chars'],
    bins=[-1, 200, 600, np.inf],
    labels=['short', 'medium', 'long']
)


def slice_metrics(df, group_col):
    rows = []
    for group, part in df.groupby(group_col, dropna=False):
        if len(part) == 0:
            continue
        rows.append({
            'group_feature': group_col,
            'group_value': str(group),
            'count': len(part),
            'phishing_rate_true': part['y_true'].mean(),
            'phishing_rate_pred': part['y_pred'].mean(),
            'accuracy': accuracy_score(part['y_true'], part['y_pred']),
            'f1': f1_score(part['y_true'], part['y_pred'], zero_division=0),
            'rmse': np.sqrt(mean_squared_error(part['y_true'], part['y_pred'])),
        })
    return pd.DataFrame(rows)

slice_tables = [
    slice_metrics(bias_df, 'has_url'),
    slice_metrics(bias_df, 'has_urgent_terms'),
    slice_metrics(bias_df, 'length_bucket'),
]

bias_report = pd.concat(slice_tables, ignore_index=True)
print('\n=== Bias / Slice Check (Ensemble) ===')
display(bias_report)


print('\n=== Disparity Summary (max-min F1 by feature) ===')
for feature in bias_report['group_feature'].unique():
    part = bias_report[bias_report['group_feature'] == feature]
    if len(part) > 1:
        gap = part['f1'].max() - part['f1'].min()
        print(f'{feature}: F1 gap = {gap:.4f}')

print('\nBias check complete. Investigate features with large F1 gaps or large predicted-vs-true phishing rate differences.')

Ignore this:

In [ ]:
%pip install "transformers==4.46.3" tf-keras

In [ ]:
%pip install sklearn